In [1]:
import json
import random
import base64
import os
import shutil
import subprocess
import dill
import nltk
import config
import itertools
import io
import math
import textwrap
import taxonomy
import time

from PIL import Image
from utils.DataHandlers import LatexDataHandler
from utils.ModelClient import ModelClient
from pathlib import Path
from collections import Counter
from pdf2image import convert_from_bytes

In [2]:
EDIT_TYPE = config.EDIT_TYPE
DATA_FOLDER_PATH = config.DATA_FOLDER_PATH
TASK_ID = 'generate_ground_truth_description'
INPUT_TYPE = 'latex'
COMPILATION_TYPE = config.COMPILATION_TYPE

In [3]:
def get_system_prompt():
    system_prompt = textwrap.dedent("""
        # Identity
        You are a helpful assistant that describes specific errors and weaknesses that are introduced into a scientific manuscript after editing.

        # Task
        Your task is to generate a description of a specified table-text error, weakness, or discrepancy that is introduced into a scientific manuscript.
                                    
        You will be provided with:
        * The latex of the original scientific manuscript.
        * The table-text error or weakness type that was introduced into the manuscript and its description.
        * A list of edits that were performed on tables to introduce the weakness into the manuscript. An edit consists of two parts: a unique span of text from the original manuscript, and a second span of text that replaces the first.
        * A list of sentences from the manuscript that demonstrate the existence of the weakness in the manuscript after the edits are performed.
        * An explanation of why the edit introduces the specified type of weakness into the manuscript.
        
        The edited manuscript can be derived from the original latex and the list of edits - it is not directly provided. You should provide a detailed description of the error or weakness and describe where in the manuscript it occurs.
    """[1:]
    )

    # error_extension = textwrap.dedent("""
    #     # Error or Weakness
    #     This section contains a list of error or weakness types and their corresponding definitions.

    #     """[1:]
    # )

    # for error_category in taxonomy.TAXONOMY_DATA.keys():
    #     description = taxonomy.TAXONOMY_DATA[error_category]['description']
    #     error_name = taxonomy.TAXONOMY_DATA[error_category]['name']
    #     error_extension += f"## {error_name}\n{description}\n\n"

    req = textwrap.dedent("""
        # Requirements
        * You should focus only on the error or weakness introduced by the edits and clarified by the provided explanation - do not mention other errors or weaknesses
        * You should be as specific as possible; try to provide the specific sentences or tables in the edited manuscript that are relevant.
        * The description should be derivable solely from the edited manuscript. Use the provided information to support your reasoning, but do not make any references to the specific edits that were performed or the original table values.
        * You should only provide the description. Do not ask follow-up questions or provide irrelevant information. 
        """[1:]
    )

    return system_prompt + req

In [4]:
SYSTEM_PROMPT = get_system_prompt()

MESSAGE_TEMPLATE = """
Manuscript Latex:
{0}
Weakness Type: 
{1}
Weakness Description: 
{2}
Edits Introduced:
{3}
Sentences Demonstrating Weakness:
{4}
Explanation of Edits:
{5}

Provide a detailed description of the weakness or error that is introduced into the manuscript by the edits.
"""



In [5]:
def manuscript_iterator(project_data):
    for paper_id, error_id, data in project_data:
        meta_id = f'{paper_id}___{error_id}' #combined paper + error id
        latex_path = DATA_FOLDER_PATH / paper_id / 'flat_main.tex'
        #paper_path = data['edited_latex_path']

        with open(latex_path, 'r') as fp:
            manuscript_latex = fp.read()
        
        weakness_type = base64.urlsafe_b64decode(error_id).decode()
        weakness_description = taxonomy.TAXONOMY_DATA[weakness_type]
        sentences = data['sentences']
        edit_list = data['edit_list']
        edit_reasoning = data['reasoning']

        message = MESSAGE_TEMPLATE.format(
            manuscript_latex,
            weakness_type,
            weakness_description,
            edit_list,
            sentences,
            edit_reasoning,
        )

        req_data = {
            'system_prompt': SYSTEM_PROMPT,
            'message': message,
            'reasoning_effort': "high",
        }
        
        yield (meta_id, req_data)

In [6]:
model_client = ModelClient("openai", "gpt-5.4-2026-03-05", "gpt_54", "output/ground_truth_gpt54_workspace")

In [7]:
with open('runtime.txt', 'w') as fp:
    ts = time.time()
    fp.write(str(ts))

In [8]:
project_ids = []
with open('./responses_edit_latex_gpt_54_compiled.dill', 'rb') as fp:
    compiled_data = dill.load(fp)

    project_data = []
    for project_id, error_id, compiled in compiled_data:
        if compiled['valid'] and compiled["edited_pdf_path"] is not None:
            project_data.append((project_id, error_id, compiled))

# manuscript_file_paths = [project_data[2] for project_data in project_ids]
cache_path = f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}_cache'

for meta_id, res in model_client.iterative_query(manuscript_iterator(project_data), cache_file_path=cache_path, batch=True):
    paper_id, error_id = meta_id.split('___')
    error_id = meta_id.split('___')[1]
    print(meta_id)
    res_dict = {
        'model_id': model_client.model_prefix_name,
        'task_id': TASK_ID,
        'response_text': res
    }

    for compiled in compiled_data:
        if compiled[0] == paper_id and compiled[1] == error_id:
            if 'ground_truth' in compiled[2]:
                compiled[2]['ground_truth'].append(res_dict)
            else:
                compiled[2]['ground_truth'] = [res_dict]
            break

with open(f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}.dill', 'wb') as fp:
    dill.dump(compiled_data, fp)

2409.14664___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
2409.14664___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=
2409.14664___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==
2409.14664___Y29udHJhZGljdGlvbl93aXRoX2VzdGFibGlzaGVkX2tub3dsZWRnZQ==
2505.13988___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
2505.13988___bG9naWNhbGx5X2ludmFsaWRfaW50ZXJwcmV0YXRpb24=
2505.13988___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=
2505.13988___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==
2505.13988___Y29udHJhZGljdGlvbl93aXRoX2VzdGFibGlzaGVkX2tub3dsZWRnZQ==
2502.10118___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
2502.10118___bG9naWNhbGx5X2ludmFsaWRfaW50ZXJwcmV0YXRpb24=
2502.10118___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=
2502.10118___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==
2502.10118___Y29udHJhZGljdGlvbl93aXRoX2VzdGFibGlzaGVkX2tub3dsZWRnZQ==
2501.09751___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0
2501.09751___bG9naWNhbGx5X2ludmFsaWRfaW50ZXJwcmV0YXRpb24=
2501.09751___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=
2501.09751___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==
2501.09751___Y29udHJhZGljdGlvbl9

In [9]:
# make sure all model responses are recorded
with open(f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}.dill', 'rb') as fp:
    responses = dill.load(fp)

print(f'Total responses: {len(responses)}')
count = 0
incomplete = 0
for mid, errid, res in responses:
    if res['valid'] and res['edited_pdf_path'] and 'ground_truth' not in res:
        count += 1

print(f'Number of missing model responses: {count}') 

Total responses: 300
Number of missing model responses: 0
